# 4. Modeling Phase: Principal Component Analysis (PCA)
## Dimensionality Reduction (CRISP-DM Target)
As an AI Engineer integrating this pipeline, PCA serves as our mathematical filter. Real-world business data often contains highly collinear variables (e.g., `Frequency` and `Total_Expenditure`). PCA projects this mathematically bloated representation down into a set of perfectly independent (orthogonal) Principal Components while preserving the maximum possible dataset variance. Our explicit ML threshold here is targeting ~90% cumulative explained variance.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

print("AI/ML Engine Initialized.")


### 4.1 Ingest the Analytical Matrix
Loading the `customer_features_scaled.csv` prepared in the previous Silver/Gold data pipeline layer.


In [ ]:
data_dir = 'data'
matrix_path = os.path.join(data_dir, 'customer_features_scaled.csv')

# Load and drop the CustomerID (we need an entirely un-labeled matrix for pure unsupervised learning)
df_scaled = pd.read_csv(matrix_path)

if 'CustomerID' in df_scaled.columns:
    df_scaled = df_scaled.set_index('CustomerID')

print(f"Matrix Ingested: {df_scaled.shape[0]} rows, {df_scaled.shape[1]} dimensional features.")
df_scaled.head()


### 4.2 Orthogonal Mathematical Transformation
We instruct Scikit-Learn to calculate the covariance matrix, its eigenvectors (the components), and eigenvalues (the variance magnitude).


In [ ]:
# Instantiate PCA without a fixed component limit to calculate full variance
pca_full = PCA()
pca_full.fit(df_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("Full PCA engine fitted successfully.")


### 4.3 Explained Variance Ratio (The Scree Plot)
The visual elbow of the Scree Plot and our explicit 90% intersection line dictates our `n_components` `k` selection.


In [ ]:
plt.figure(figsize=(10, 6))

# Individual Variance Bar Chart
plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.6, align='center',
        label='Individual explained variance', color='teal')
# Cumulative Variance Step Plot
plt.step(range(1, len(cumulative_variance) + 1), cumulative_variance, where='mid',
         label='Cumulative explained variance', color='crimson')

plt.axhline(y=0.90, color='r', linestyle='--', label='90% Variance Threshold')

plt.ylabel('Explained Variance Ratio')
plt.xlabel('Principal Component Index')
plt.title('PCA Explained Variance / Scree Plot')
plt.legend(loc='best')
plt.tight_layout()
plt.show()


### 4.4 Transforming Down using `k` Components
We programmatically evaluate how many components it takes to safely cross the 0.90 threshold, build a hard-coded PCA filter, and transform.


In [ ]:
# Find N where cumulative variance >= 0.90
target_k = np.argmax(cumulative_variance >= 0.90) + 1
print(f"Optimal number of Principal Components to retain >= 90% variance: k={target_k}")

# Re-fit our ML model with strict boundaries
pca_final = PCA(n_components=target_k)
pca_features = pca_final.fit_transform(df_scaled)

# Store inside a dataframe dynamically
pc_columns = [f"PC{i+1}" for i in range(target_k)]
pca_df = pd.DataFrame(data=pca_features, columns=pc_columns, index=df_scaled.index)

print(f"Dataset successfully compressed from {df_scaled.shape[1]} to {target_k} components.")
pca_df.head()


### 4.5 Dimensional Score Plot
Let's map Principal Component 1 against Principal Component 2 to see if natural topological behavior clusters are natively separating ahead of K-Means integration.


In [ ]:
plt.figure(figsize=(10, 8))

# Scatter the first two components
sns.scatterplot(x='PC1', y='PC2', data=pca_df, alpha=0.7, color='midnightblue')

plt.title('Customer Behavioral Scatter (PC1 vs PC2)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.tight_layout()
plt.show()


### 4.6 Deployment to Disk
The final modeling matrix is now rendered down to completely orthogonal signals and completely stripped of white noise. This is explicitly verified by our AI Engineer logic and is now ready for deployment to the `kmeans.ipynb` phase.


In [ ]:
export_path = os.path.join(data_dir, 'customer_pca_features.csv')
pca_df.to_csv(export_path)
print(f"Reduced PCA dataset securely exported to {export_path}")
